# Multi-Agent Sepsis Prediction Model Training

This notebook trains the multi-agent sepsis prediction system using the processed MIMIC-IV data.

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    MULTI-AGENT SYSTEM                       │
├─────────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐      │
│  │ Vitals Agent │  │  Labs Agent  │  │ Trend Agent  │      │
│  │  (LSTM+Attn) │  │ (LSTM+Imput) │  │ (Transformer)│      │
│  └──────┬───────┘  └──────┬───────┘  └──────┬───────┘      │
│         └────────────┬────┴─────────────────┘               │
│                      ▼                                      │
│              ┌───────────────┐                              │
│              │  Meta-Learner │                              │
│              └───────┬───────┘                              │
│                      ▼                                      │
│              Sepsis Probability                             │
└─────────────────────────────────────────────────────────────┘
```

**Author:** Jason  
**Date:** February 2026

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm h5py

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, 
    precision_recall_curve, roc_curve,
    confusion_matrix, classification_report
)

# Paths
PROJECT_PATH = "/content/drive/MyDrive/Sepsis"
DATA_PATH = f"{PROJECT_PATH}/data/processed/mimic_harmonized"
MODEL_PATH = f"{PROJECT_PATH}/models"

# Add src to path
sys.path.append(f"{PROJECT_PATH}/src")

# Create model directory
os.makedirs(MODEL_PATH, exist_ok=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import our multi-agent model
from models.multi_agent import (
    MultiAgentSepsisPredictor, 
    FocalLoss, 
    count_parameters
)

print("Multi-agent model imported successfully!")

## 2. Configuration

In [ ]:
# ========================================
# TRAINING CONFIGURATION
# ========================================

CONFIG = {
    # Data
    'data_file': f"{DATA_PATH}/mimic_processed_medium.h5",
    'sequence_length': 24,  # Hours of history to use
    'test_size': 0.2,
    'val_size': 0.1,
    'random_seed': 42,
    
    # Model
    'hidden_dim': 64,
    'num_layers': 2,
    'dropout': 0.3,
    
    # Training
    'batch_size': 64,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'epochs': 50,
    'patience': 10,  # Early stopping patience
    
    # Loss
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
    
    # Feature groups
    'vitals_features': ['hr', 'resp', 'temp', 'sbp', 'dbp', 'map_value', 'o2sat'],
    'labs_features': ['bun', 'chloride', 'creatinine', 'wbc', 'bicarbonate', 'platelets',
                      'magnesium', 'calcium', 'potassium', 'sodium', 'glucose',
                      'fio2', 'ph', 'paco2', 'pao2', 'lactate', 'bilirubin'],
}

# All features for trend agent
CONFIG['all_features'] = CONFIG['vitals_features'] + CONFIG['labs_features']

print("Configuration:")
for key, value in CONFIG.items():
    if not isinstance(value, list):
        print(f"  {key}: {value}")
print(f"  vitals_features: {len(CONFIG['vitals_features'])} features")
print(f"  labs_features: {len(CONFIG['labs_features'])} features")
print(f"  all_features: {len(CONFIG['all_features'])} features")

## 3. Load and Prepare Data

In [ ]:
# Load processed data
print("Loading data...")
df = pd.read_hdf(CONFIG['data_file'], key='data')

print(f"\nDataset loaded:")
print(f"  Observations: {len(df):,}")
print(f"  Patients: {df['subject_id'].nunique()}")
print(f"  Features: {len(df.columns)}")

In [ ]:
# Check available features
print("\nChecking features...")

# Vitals
vitals_available = [f for f in CONFIG['vitals_features'] if f in df.columns]
vitals_missing = [f for f in CONFIG['vitals_features'] if f not in df.columns]
print(f"\nVitals: {len(vitals_available)}/{len(CONFIG['vitals_features'])} available")
if vitals_missing:
    print(f"  Missing: {vitals_missing}")

# Labs
labs_available = [f for f in CONFIG['labs_features'] if f in df.columns]
labs_missing = [f for f in CONFIG['labs_features'] if f not in df.columns]
print(f"\nLabs: {len(labs_available)}/{len(CONFIG['labs_features'])} available")
if labs_missing:
    print(f"  Missing: {labs_missing}")

# Update config with available features
CONFIG['vitals_features'] = vitals_available
CONFIG['labs_features'] = labs_available
CONFIG['all_features'] = vitals_available + labs_available

print(f"\nUsing {len(CONFIG['all_features'])} total features")

In [ ]:
# Normalize features
print("\nNormalizing features...")

feature_stats = {}
for feature in CONFIG['all_features']:
    mean = df[feature].mean()
    std = df[feature].std()
    if std == 0 or pd.isna(std):
        std = 1.0
    feature_stats[feature] = {'mean': mean, 'std': std}
    df[feature] = (df[feature] - mean) / std

print(f"Normalized {len(feature_stats)} features")

# Save normalization stats for inference
with open(f"{MODEL_PATH}/feature_stats.json", 'w') as f:
    json.dump({k: {kk: float(vv) for kk, vv in v.items()} for k, v in feature_stats.items()}, f, indent=2)
print(f"Saved normalization stats to {MODEL_PATH}/feature_stats.json")

In [ ]:
# Split by patient (prevent data leakage)
print("\nSplitting data by patient...")

patient_ids = df['subject_id'].unique()
np.random.seed(CONFIG['random_seed'])
np.random.shuffle(patient_ids)

# Get patient-level sepsis labels for stratified split
patient_labels = df.groupby('subject_id')['sepsis_label'].max().loc[patient_ids]

# First split: train+val vs test
train_val_ids, test_ids = train_test_split(
    patient_ids, 
    test_size=CONFIG['test_size'],
    stratify=patient_labels,
    random_state=CONFIG['random_seed']
)

# Second split: train vs val
train_val_labels = patient_labels.loc[train_val_ids]
train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=CONFIG['val_size'] / (1 - CONFIG['test_size']),
    stratify=train_val_labels,
    random_state=CONFIG['random_seed']
)

print(f"\nPatient splits:")
print(f"  Train: {len(train_ids)} patients")
print(f"  Val: {len(val_ids)} patients")
print(f"  Test: {len(test_ids)} patients")

# Create dataframes
train_df = df[df['subject_id'].isin(train_ids)].copy()
val_df = df[df['subject_id'].isin(val_ids)].copy()
test_df = df[df['subject_id'].isin(test_ids)].copy()

print(f"\nObservation splits:")
print(f"  Train: {len(train_df):,} observations ({train_df['sepsis_label'].mean()*100:.1f}% positive)")
print(f"  Val: {len(val_df):,} observations ({val_df['sepsis_label'].mean()*100:.1f}% positive)")
print(f"  Test: {len(test_df):,} observations ({test_df['sepsis_label'].mean()*100:.1f}% positive)")

## 4. Create PyTorch Dataset

In [ ]:
class SepsisSequenceDataset(Dataset):
    """
    PyTorch Dataset for sepsis prediction.
    
    Creates sequences of patient data for temporal modeling.
    Each sample is a window of `seq_length` hours ending at the current time.
    """
    
    def __init__(
        self, 
        df: pd.DataFrame,
        vitals_features: list,
        labs_features: list,
        seq_length: int = 24
    ):
        self.seq_length = seq_length
        self.vitals_features = vitals_features
        self.labs_features = labs_features
        self.all_features = vitals_features + labs_features
        
        # Group by patient and create sequences
        self.sequences = []
        self.labels = []
        self.patient_ids = []
        
        print("Creating sequences...")
        for patient_id, group in tqdm(df.groupby('subject_id')):
            group = group.sort_values('charttime').reset_index(drop=True)
            
            # Skip if not enough data
            if len(group) < seq_length:
                continue
            
            # Create sliding windows
            for i in range(seq_length, len(group) + 1):
                # Get sequence
                seq = group.iloc[i-seq_length:i]
                
                # Get label (from last timestep)
                label = seq['sepsis_label'].iloc[-1]
                
                # Get features
                features = seq[self.all_features].values
                
                self.sequences.append(features)
                self.labels.append(label)
                self.patient_ids.append(patient_id)
        
        self.sequences = np.array(self.sequences, dtype=np.float32)
        self.labels = np.array(self.labels, dtype=np.float32)
        
        # Replace NaN with 0 and create missing masks
        self.missing_mask = np.isnan(self.sequences)
        self.sequences = np.nan_to_num(self.sequences, nan=0.0)
        
        print(f"Created {len(self.sequences):,} sequences")
        print(f"Positive rate: {self.labels.mean()*100:.1f}%")
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        # Split features into vitals and labs
        n_vitals = len(self.vitals_features)
        
        vitals = self.sequences[idx, :, :n_vitals]
        labs = self.sequences[idx, :, n_vitals:]
        labs_mask = self.missing_mask[idx, :, n_vitals:]
        all_features = self.sequences[idx]
        label = self.labels[idx]
        
        return {
            'vitals': torch.tensor(vitals, dtype=torch.float32),
            'labs': torch.tensor(labs, dtype=torch.float32),
            'labs_mask': torch.tensor(labs_mask, dtype=torch.float32),
            'all_features': torch.tensor(all_features, dtype=torch.float32),
            'label': torch.tensor(label, dtype=torch.float32).unsqueeze(0)
        }

In [ ]:
# Create datasets
print("Creating training dataset...")
train_dataset = SepsisSequenceDataset(
    train_df, 
    CONFIG['vitals_features'],
    CONFIG['labs_features'],
    CONFIG['sequence_length']
)

print("\nCreating validation dataset...")
val_dataset = SepsisSequenceDataset(
    val_df,
    CONFIG['vitals_features'],
    CONFIG['labs_features'],
    CONFIG['sequence_length']
)

print("\nCreating test dataset...")
test_dataset = SepsisSequenceDataset(
    test_df,
    CONFIG['vitals_features'],
    CONFIG['labs_features'],
    CONFIG['sequence_length']
)

In [ ]:
# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\nData loaders created:")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val: {len(val_loader)} batches")
print(f"  Test: {len(test_loader)} batches")

## 5. Initialize Model

In [ ]:
# Initialize model
model = MultiAgentSepsisPredictor(
    vitals_dim=len(CONFIG['vitals_features']),
    labs_dim=len(CONFIG['labs_features']),
    all_features_dim=len(CONFIG['all_features']),
    hidden_dim=CONFIG['hidden_dim'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout']
).to(device)

print(f"\nModel initialized:")
print(f"  Parameters: {count_parameters(model):,}")
print(f"  Device: {device}")

# Loss function
criterion = FocalLoss(alpha=CONFIG['focal_alpha'], gamma=CONFIG['focal_gamma'])

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# Learning rate scheduler (reduces LR when validation AUROC plateaus)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

print("Optimizer and scheduler ready!")

## 6. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(loader, desc="Training", leave=False):
        # Move to device
        vitals = batch['vitals'].to(device)
        labs = batch['labs'].to(device)
        labs_mask = batch['labs_mask'].to(device)
        all_features = batch['all_features'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        output = model(vitals, labs, labs_mask, all_features)
        
        # Loss
        loss = criterion(output['logits'], labels)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        all_preds.extend(output['probability'].detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
    
    # Calculate metrics
    avg_loss = total_loss / len(loader)
    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    
    auroc = roc_auc_score(all_labels, all_preds)
    auprc = average_precision_score(all_labels, all_preds)
    
    return avg_loss, auroc, auprc


def evaluate(model, loader, criterion, device):
    """Evaluate on validation/test set."""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_agent_weights = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            vitals = batch['vitals'].to(device)
            labs = batch['labs'].to(device)
            labs_mask = batch['labs_mask'].to(device)
            all_features = batch['all_features'].to(device)
            labels = batch['label'].to(device)
            
            output = model(vitals, labs, labs_mask, all_features)
            loss = criterion(output['logits'], labels)
            
            total_loss += loss.item()
            all_preds.extend(output['probability'].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_agent_weights.extend(output['agent_weights'].cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    
    auroc = roc_auc_score(all_labels, all_preds)
    auprc = average_precision_score(all_labels, all_preds)
    
    return avg_loss, auroc, auprc, all_preds, all_labels, np.array(all_agent_weights)

In [ ]:
# Training loop with early stopping
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70 + "\n")

best_val_auroc = 0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_auroc': [], 'val_auroc': [], 'train_auprc': [], 'val_auprc': []}

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
    print("-" * 40)
    
    # Train
    train_loss, train_auroc, train_auprc = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validate
    val_loss, val_auroc, val_auprc, _, _, _ = evaluate(
        model, val_loader, criterion, device
    )
    
    # Update scheduler
    scheduler.step(val_auroc)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auroc'].append(train_auroc)
    history['val_auroc'].append(val_auroc)
    history['train_auprc'].append(train_auprc)
    history['val_auprc'].append(val_auprc)
    
    # Print metrics
    print(f"  Train - Loss: {train_loss:.4f}, AUROC: {train_auroc:.4f}, AUPRC: {train_auprc:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, AUROC: {val_auroc:.4f}, AUPRC: {val_auprc:.4f}")
    
    # Check for improvement
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        patience_counter = 0
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auroc': val_auroc,
            'config': CONFIG
        }, f"{MODEL_PATH}/best_model.pt")
        print(f"  ✓ New best model saved! (AUROC: {val_auroc:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{CONFIG['patience']})")
    
    # Early stopping
    if patience_counter >= CONFIG['patience']:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

print("\n" + "="*70)
print(f"TRAINING COMPLETE - Best Val AUROC: {best_val_auroc:.4f}")
print("="*70)

## 7. Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUROC
axes[1].plot(history['train_auroc'], label='Train', linewidth=2)
axes[1].plot(history['val_auroc'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUROC')
axes[1].set_title('AUROC Over Training', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# AUPRC
axes[2].plot(history['train_auprc'], label='Train', linewidth=2)
axes[2].plot(history['val_auprc'], label='Validation', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUPRC')
axes[2].set_title('AUPRC Over Training', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/training_history.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Test Set Evaluation

In [ ]:
# Load best model (weights_only=False needed for PyTorch 2.6+)
checkpoint = torch.load(f"{MODEL_PATH}/best_model.pt", weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

# Evaluate on test set
test_loss, test_auroc, test_auprc, test_preds, test_labels, agent_weights = evaluate(
    model, test_loader, criterion, device
)

print("\n" + "="*70)
print("TEST SET RESULTS")
print("="*70)
print(f"\n  AUROC: {test_auroc:.4f}")
print(f"  AUPRC: {test_auprc:.4f}")
print(f"  Loss:  {test_loss:.4f}")

In [ ]:
# Find optimal threshold
precision, recall, thresholds_pr = precision_recall_curve(test_labels, test_preds)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds_pr[optimal_idx] if optimal_idx < len(thresholds_pr) else 0.5

print(f"\nOptimal Threshold (F1): {optimal_threshold:.3f}")
print(f"  Precision: {precision[optimal_idx]:.3f}")
print(f"  Recall: {recall[optimal_idx]:.3f}")
print(f"  F1 Score: {f1_scores[optimal_idx]:.3f}")

In [ ]:
# Classification report with optimal threshold
test_preds_binary = (test_preds >= optimal_threshold).astype(int)

print("\nClassification Report:")
print(classification_report(test_labels, test_preds_binary, target_names=['No Sepsis', 'Sepsis']))

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds_binary)
print("\nConfusion Matrix:")
print(f"  TN: {cm[0,0]:,}  FP: {cm[0,1]:,}")
print(f"  FN: {cm[1,0]:,}  TP: {cm[1,1]:,}")

In [ ]:
# Plot ROC and PR curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ROC Curve
fpr, tpr, _ = roc_curve(test_labels, test_preds)
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUROC = {test_auroc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# PR Curve
axes[1].plot(recall, precision, 'g-', linewidth=2, label=f'AUPRC = {test_auprc:.3f}')
axes[1].axhline(y=test_labels.mean(), color='k', linestyle='--', linewidth=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['No Sepsis', 'Sepsis'],
            yticklabels=['No Sepsis', 'Sepsis'])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/test_results.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Agent Contribution Analysis

In [ ]:
# Analyze agent contributions
avg_weights = agent_weights.mean(axis=0)
agent_names = ['Vitals Agent', 'Labs Agent', 'Trend Agent']

print("\nAgent Contributions:")
for name, weight in zip(agent_names, avg_weights):
    print(f"  {name}: {weight*100:.1f}%")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
colors = ['#3498db', '#2ecc71', '#e74c3c']
axes[0].pie(avg_weights, labels=agent_names, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Average Agent Contributions', fontweight='bold')

# Bar chart by prediction
sepsis_weights = agent_weights[test_labels == 1].mean(axis=0)
non_sepsis_weights = agent_weights[test_labels == 0].mean(axis=0)

x = np.arange(len(agent_names))
width = 0.35

bars1 = axes[1].bar(x - width/2, non_sepsis_weights*100, width, label='No Sepsis', color='#3498db')
bars2 = axes[1].bar(x + width/2, sepsis_weights*100, width, label='Sepsis', color='#e74c3c')

axes[1].set_ylabel('Contribution (%)')
axes[1].set_title('Agent Contributions by Outcome', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(agent_names)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/agent_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Final Results

In [ ]:
# Save final results
results = {
    'test_auroc': float(test_auroc),
    'test_auprc': float(test_auprc),
    'test_loss': float(test_loss),
    'optimal_threshold': float(optimal_threshold),
    'best_epoch': int(checkpoint['epoch']),
    'agent_contributions': {
        'vitals': float(avg_weights[0]),
        'labs': float(avg_weights[1]),
        'trend': float(avg_weights[2])
    },
    'confusion_matrix': cm.tolist(),
    'config': {k: v for k, v in CONFIG.items() if not isinstance(v, list)},
    'training_date': datetime.now().isoformat()
}

with open(f"{MODEL_PATH}/results.json", 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to:", f"{MODEL_PATH}/results.json")

# Save training history
with open(f"{MODEL_PATH}/history.json", 'w') as f:
    json.dump(history, f, indent=2)

print("History saved to:", f"{MODEL_PATH}/history.json")

In [ ]:
# Summary
print("\n" + "="*70)
print("TRAINING COMPLETE - SUMMARY")
print("="*70)

print(f"\nModel: Multi-Agent Sepsis Predictor")
print(f"  Parameters: {count_parameters(model):,}")
print(f"  Best Epoch: {checkpoint['epoch']+1}")

print(f"\nTest Performance:")
print(f"  AUROC: {test_auroc:.4f}")
print(f"  AUPRC: {test_auprc:.4f}")

print(f"\nAgent Contributions:")
for name, weight in zip(agent_names, avg_weights):
    print(f"  {name}: {weight*100:.1f}%")

print(f"\nFiles saved:")
print(f"  - {MODEL_PATH}/best_model.pt")
print(f"  - {MODEL_PATH}/results.json")
print(f"  - {MODEL_PATH}/history.json")
print(f"  - {MODEL_PATH}/feature_stats.json")
print(f"  - {MODEL_PATH}/training_history.png")
print(f"  - {MODEL_PATH}/test_results.png")
print(f"  - {MODEL_PATH}/agent_analysis.png")

print("\n" + "="*70)

## Next Steps

Now you have a trained multi-agent sepsis prediction model!

### What you can do next:

1. **Inference on new patients**: Use the saved model to predict sepsis risk
2. **Hyperparameter tuning**: Try different hidden_dim, num_layers, etc.
3. **Train on more data**: Process "large" mode (5,000 patients) for better performance
4. **External validation**: Test on CinC 2019 challenge data
5. **Explainability**: Analyze which features drive predictions

### To use the model for inference:
```python
# Load model
model = MultiAgentSepsisPredictor(...)
model.load_state_dict(torch.load('best_model.pt')['model_state_dict'])
model.eval()

# Predict
with torch.no_grad():
    output = model(vitals, labs, labs_mask, all_features)
    risk = output['probability'].item()
    print(f"Sepsis Risk: {risk*100:.1f}%")
```